# Task 3 — Audio tagging (Colab GPU training)

This notebook trains the Task 3 CNN on a Colab Pro+ GPU and produces `predictions3.json`.

**Before running:**
1. Upload `student_files_updated.zip` (the original data zip) to your Google Drive — anywhere works; the default expected path is `/content/drive/MyDrive/CSE153/student_files_updated.zip`.
2. Set runtime → GPU (A100 / V100 / T4 all work).
3. Run the cells top-to-bottom.

At the end, `predictions3.json` is saved both to the Colab working directory (downloadable from the file panel) and to your Drive at `/content/drive/MyDrive/CSE153/predictions3.json`.

## 1. Verify GPU + install missing deps

In [ ]:
!nvidia-smi

In [ ]:
# Colab already has torch / torchaudio / librosa / sklearn / numpy.
# Install nothing else for Task 3.
import torch, torchaudio, librosa, sklearn, numpy as np
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 2. Mount Google Drive and unzip the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil
ZIP_PATH = '/content/drive/MyDrive/CSE153/student_files_updated.zip'
WORKDIR = '/content/work'
assert os.path.exists(ZIP_PATH), f'Zip not found at {ZIP_PATH} — upload it to Drive or change ZIP_PATH'
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
if not os.path.isdir('student_files'):
    !unzip -q -o {ZIP_PATH} -d {WORKDIR}
    if os.path.isdir(os.path.join(WORKDIR, '__MACOSX')):
        shutil.rmtree(os.path.join(WORKDIR, '__MACOSX'))
print('Contents:', os.listdir('student_files'))
print('Train clips:', len(os.listdir('student_files/task3_audio_classification/train')))
print('Test clips:', len(os.listdir('student_files/task3_audio_classification/test')))

## 3. Training code (CNN with SpecAugment, BCEWithLogitsLoss, AdamW + cosine)

Same architecture used locally, scaled up for GPU: larger batch size, optional larger n_mels / hop, more epochs.

In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import librosa
from torch.utils.data import Dataset, DataLoader, random_split
from torchaudio.transforms import MelSpectrogram, AmplitudeToDB
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm

DATAROOT = 'student_files/task3_audio_classification'
SAMPLE_RATE = 22050
N_MELS = 96
N_CLASSES = 10
AUDIO_DURATION = 10
BATCH_SIZE = 64           # GPU-friendly
EPOCHS = 30
LR = 1e-3
TAGS = ['rock', 'oldies', 'jazz', 'pop', 'dance', 'blues', 'punk', 'chill', 'electronic', 'country']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

In [ ]:
def extract_waveform(path):
    waveform, sr = librosa.load(os.path.join(DATAROOT, path), sr=SAMPLE_RATE)
    w = torch.FloatTensor(np.array([waveform]))
    tlen = SAMPLE_RATE * AUDIO_DURATION
    if w.shape[1] < tlen:
        w = F.pad(w, (0, tlen - w.shape[1]))
    else:
        w = w[:, :tlen]
    return w

class AudioDataset(Dataset):
    def __init__(self, meta):
        self.meta = meta
        self.ids = list(meta.keys())
        self.mel = MelSpectrogram(sample_rate=SAMPLE_RATE, n_mels=N_MELS, n_fft=1024, hop_length=256, f_min=20, f_max=SAMPLE_RATE // 2)
        self.db = AmplitudeToDB(top_db=80)
        self.feats = {}
        for p in tqdm(self.ids, desc='Preloading mels'):
            w = extract_waveform(p)
            m = self.db(self.mel(w)).squeeze(0)
            m = (m - m.mean()) / (m.std() + 1e-6)
            self.feats[p] = m
    def _aug(self, m):
        if random.random() < 0.5:
            f = random.randint(0, 16); f0 = random.randint(0, max(1, N_MELS - f))
            m[f0:f0+f, :] = 0
        if random.random() < 0.5:
            t = random.randint(0, 40); t0 = random.randint(0, max(1, m.shape[1] - t))
            m[:, t0:t0+t] = 0
        return m
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, idx):
        p = self.ids[idx]
        lab = torch.tensor([1 if t in self.meta[p] else 0 for t in TAGS], dtype=torch.float32)
        return self.feats[p].unsqueeze(0), lab, p

class AudSubset(Dataset):
    def __init__(self, subset, augment):
        self.dataset = subset.dataset; self.indices = subset.indices; self.augment = augment
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        p = self.dataset.ids[real_idx]
        lab = torch.tensor([1 if t in self.dataset.meta[p] else 0 for t in TAGS], dtype=torch.float32)
        m = self.dataset.feats[p].clone()
        if self.augment:
            m = self.dataset._aug(m)
        return m.unsqueeze(0), lab, p

In [ ]:
class CNNClassifier(nn.Module):
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        def block(ic, oc, pool=(2, 4)):
            return nn.Sequential(
                nn.Conv2d(ic, oc, 3, padding=1, bias=False),
                nn.BatchNorm2d(oc), nn.ReLU(inplace=True),
                nn.Conv2d(oc, oc, 3, padding=1, bias=False),
                nn.BatchNorm2d(oc), nn.ReLU(inplace=True),
                nn.MaxPool2d(pool),
            )
        self.b1 = block(1, 32)
        self.b2 = block(32, 64)
        self.b3 = block(64, 128)
        self.b4 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(256, n_classes)
    def forward(self, x):
        x = self.b1(x); x = self.b2(x); x = self.b3(x); x = self.b4(x)
        x = x.view(x.size(0), -1)
        return self.fc(self.dropout(x))

def evaluate(model, loader):
    model.eval(); all_logits=[]; all_targets=[]; all_paths=[]
    with torch.no_grad():
        for x, y, ps in loader:
            x = x.to(DEVICE); y = y.to(DEVICE)
            all_logits.append(model(x).cpu()); all_targets.append(y.cpu()); all_paths += list(ps)
    logits = torch.cat(all_logits); targets = torch.cat(all_targets)
    probs = torch.sigmoid(logits).numpy(); tn = targets.numpy()
    mAP = None
    if tn.sum() > 0:
        try:
            mAP = average_precision_score(tn, probs, average='macro')
        except Exception:
            mAP = None
    return probs, all_paths, mAP

In [ ]:
torch.manual_seed(0); random.seed(0); np.random.seed(0)

train_meta = eval(open(os.path.join(DATAROOT, 'train.json')).read())
test_meta = {k: [] for k in eval(open(os.path.join(DATAROOT, 'test.json')).read())}
print('Train:', len(train_meta), 'Test:', len(test_meta))

all_train = AudioDataset(train_meta)
g = torch.Generator().manual_seed(0)
n = len(all_train); n_tr = int(n * 0.9); n_va = n - n_tr
tr_sub, va_sub = random_split(all_train, [n_tr, n_va], generator=g)
tr = AudSubset(tr_sub, augment=True)
va = AudSubset(va_sub, augment=False)
te = AudioDataset(test_meta)

loader_tr = DataLoader(tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
loader_va = DataLoader(va, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
loader_te = DataLoader(te, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

## 4. Train

In [ ]:
model = CNNClassifier().to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
crit = nn.BCEWithLogitsLoss()
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

best_map = -1.0; best_state = None
for ep in range(EPOCHS):
    model.train(); running = 0.0; nb = 0
    for x, y, _ in tqdm(loader_tr, desc=f'Epoch {ep+1}/{EPOCHS}'):
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        opt.zero_grad()
        loss = crit(model(x), y); loss.backward(); opt.step()
        running += loss.item(); nb += 1
    sched.step()
    _, _, vmap = evaluate(model, loader_va)
    print(f'[ep{ep+1}] loss={running/nb:.4f} val_mAP={vmap:.4f}')
    if vmap is not None and vmap > best_map:
        best_map = vmap
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
print('Best val mAP:', best_map)

## 5. Generate predictions3.json

In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)
probs, paths, _ = evaluate(model, loader_te)
preds = {(p[2:] if p.startswith('./') else p): {TAGS[j]: float(probs[i][j]) for j in range(N_CLASSES)} for i, p in enumerate(paths)}

out_local = '/content/work/predictions3.json'
with open(out_local, 'w') as f:
    f.write(repr(preds) + '\n')
print('Wrote', out_local, 'entries=', len(preds))

# Also copy to Drive for easy download
import shutil
drive_out = '/content/drive/MyDrive/CSE153/predictions3.json'
os.makedirs(os.path.dirname(drive_out), exist_ok=True)
shutil.copy(out_local, drive_out)
print('Saved copy to', drive_out)

In [ ]:
# Quick sanity check on the output format
import os
d = eval(open(out_local).read())
print('len:', len(d))
sample_k = list(d.keys())[0]
print('sample key:', repr(sample_k))
print('sample value:', d[sample_k])
assert all(isinstance(v, dict) and len(v) == N_CLASSES for v in d.values()), 'malformed predictions'
print('OK')

Download `/content/work/predictions3.json` from the Colab file panel (or just pull it from your Drive at `/content/drive/MyDrive/CSE153/predictions3.json`). Place it alongside `predictions1.json` and `predictions2.json` in the assignment directory and submit all three.